In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
from astropy.time import Time

%matplotlib ipympl

In [ ]:
from lsst.daf.butler import Butler

In [ ]:
from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.summit.utils import BestEffortIsr


In [ ]:

#from lsst.ts.observing.utilities.auxtel.latiss.getters import get_image
#from lsst.ts.observing.utilities.auxtel.latiss.utils import parse_visit_id

def parse_visit_id(visit_id):
    """Return a data_id dictionary from a visit ID (which is returned from
    the take_image command to the ATCamera)
    The dictionary is formatted for a gen3 butler.

    Parameters
    ----------
    visit_id: `int` or `str`
        Visit id (e.g. '2021032300308')

    Returns
    -------
    data_id: `dict`
        dictionary with newly derived day_obs and seq_num keys to be
        used with a butler
    """
    _visit_id = str(visit_id)
    day_obs = int(f"{_visit_id[0:4]}{_visit_id[4:6]}{_visit_id[6:8]}")
    seq_num = int(_visit_id[9::])

    data_id = {
        "day_obs": day_obs,
        "seq_num": seq_num,
        "detector": 0,
        "instrument": "LATISS",
    }

    return data_id

In [ ]:

class Auxtel:

    def __init__(self):

        logger = logging.getLogger(f"TCBP scan {Time.now()} UT")
        logger.level = logging.INFO

        # Instantiate the control classes
        domain = salobj.Domain()
        self.atcs = ATCS(domain)
        self.atcs.set_rem_loglevel(logging.INFO)
        self.latiss = LATISS(domain)
        self.latiss.set_rem_loglevel(logging.INFO)
        #await asyncio.gather(self.atcs.start_task, self.latiss.start_task)

        print('auxtel initialization done')

        # Get instrument set up
        # inst_setup = await self.latiss.get_available_instrument_setup()
        # logger.info(f'LATISS filters are: {inst_setup[0]}')
        # logger.info(f'LATISS gratings are: {inst_setup[1]}')


    def take_auxtel_image(self,img_type, exptime, n_exp, reason, program):
        self.image_id = self.latiss.take_imgtype(imgtype=img_type,
                                exptime=exptime, 
                                n=n_exp, 
                                reason=reason, 
                                program=program)
        
        print('auxtel acquisition done')
        return self.image_id
        

    def get_auxtel_image(self, dataId=''):
        if dataId == '':
            dataIdd = parse_visit_id(str(self.image_id[0]))
        best_effort_isr = BestEffortIsr()
        exp = best_effort_isr.getExposure(dataId, detector=0)
        image = exp.getImage().array
        return image
        



In [ ]:
#from auxtel import Auxtel

auxtel_class = Auxtel()



In [ ]:
# auxtel settings
expo_time_auxtel = 2
img_type = "CBP"
n_expo = 1
reason = 'scan'
program = 'scan'



dataId = await auxtel_class.take_auxtel_image(img_type, expo_time_auxtel, n_expo, reason, program)

In [ ]:
parse_visit_id(dataId[0])

In [ ]:
image = auxtel_class.get_auxtel_image(dataId=parse_visit_id(dataId[0]))

In [ ]:
fig = plt.figure()
plt.imshow(image, origin="lower")
plt.show()

In [ ]:
butler = Butler("LATISS", collections=["LATISS/raw/all", "LATISS/calib/legacy", "LATISS/calib"])
#where = "instrument='LATISS' AND (exposure.day_obs>=20260503) AND (exposure.day_obs<20260504) AND exposure.observation_type='cbp'"
where = "instrument='LATISS' AND (exposure.id>=2026050300029) AND (exposure.id<2026050300040) AND exposure.observation_type='cbp'"
records = list(butler.registry.queryDimensionRecords('exposure', datasets='raw', where=where))
records

In [ ]:
def get_auxtel_images(dataIds):
    best_effort_isr = BestEffortIsr()
    exps = [best_effort_isr.getExposure(parse_visit_id(dataId), detector=0) for dataId in dataIds]
    images = [exp.getImage().array for exp in exps]
    return images


In [ ]:
dataIds = np.arange(2026050300004, 2026050300038, 1, dtype=int)

images = get_auxtel_images(dataIds)

In [ ]:
from scipy import signal
import cv2
from astropy.io import fits
import os

def calculate_laplacian_focus(image, type=cv2.CV_32F):
    laplacian = cv2.Laplacian(image, type)
    variance = laplacian.var()
    return variance, laplacian

def calculate_brenner_measure(image):
    """
    Brenner's focus measure.

    Parameters
    ----------
    image : np.ndarray
        The input image in grayscale.
    Returns
    -------
    float
        The Brenner value.
    """

    # Get the dimensions of the image
    height, width = image.shape

    # Initialize two matrices for horizontal and vertical focus measures
    horizontal_diff = np.zeros((height, width))
    vertical_diff = np.zeros((height, width))

    # Calculate horizontal and vertical focus measures
    horizontal_diff[:, : width - 2] = np.clip(
        image[:, 2:] - image[:, :-2], 0, None
    )
    vertical_diff[: height - 2, :] = np.clip(
        image[2:, :] - image[:-2, :], 0, None
    )

    # Calculate final focus measure
    Brenner_measure = np.max((horizontal_diff, vertical_diff), axis=0) ** 2

    # Convert focus measure matrix to 8-bit for visualization
    #Brenner_image = ((Brenner_measure / Brenner_measure.max()) * 255).astype(np.uint8)

    return Brenner_measure.mean()

In [ ]:
xspot, yspot = 2447, 1400
focuser = [73000, 72000] + list(np.arange(74000, 83000, 1000, dtype=int)) + list(np.arange(72000, 82000, 1000, dtype=int)[::-1]) + list(np.arange(75400, 78000, 200, dtype=int))

allBrenner_measures = []
allVar_Laplacian = []
for i in range(len(images)):
    image = images[i]
    dpix = 100
    image_cut = image[yspot-dpix:yspot+dpix, xspot-dpix:xspot+dpix]

    Brenner_measure = calculate_brenner_measure(image_cut)
    allBrenner_measures.append(Brenner_measure)

    variance, _ = calculate_laplacian_focus(image_cut)
    allVar_Laplacian.append(variance)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(focuser, allBrenner_measures, 'o-')
plt.title("Brenner Measure vs Focuser Position")
plt.xlabel("Focuser Position")
plt.ylabel("Brenner Measure")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(focuser, allVar_Laplacian, 'o-')
plt.title("Laplacian Variance vs Focuser Position")
plt.xlabel("Focuser Position")
plt.ylabel("Laplacian Variance")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure()
plt.imshow(images[25], origin='lower')
plt.colorbar()
plt.show()